# 🏋️ AI Exercise Detection — Google Colab Setup

This notebook sets up the entire project on Google Colab:
1. Install dependencies
2. Mount Google Drive (optional)
3. Clone / upload the repository
4. Generate synthetic dataset
5. Train the model
6. Run a quick inference demo

## 1. Install Dependencies

In [ ]:
!pip install -q mediapipe opencv-python numpy pandas scikit-learn matplotlib tqdm joblib

## 2. Mount Google Drive (Optional)

Mount Drive if you want to save the trained model or dataset permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Optional: set project root inside Drive
# PROJECT_ROOT = '/content/drive/MyDrive/AI-Exercise-Detection'
# !mkdir -p {PROJECT_ROOT}

## 3. Get the Repository

**Option A:** Clone from GitHub (replace URL with your repo)

In [ ]:
# Option A: Clone from GitHub
# !git clone https://github.com/your-username/AI-Exercise-Detection.git

# Option B: If you've uploaded a zip
# !unzip AI-Exercise-Detection.zip

import os
os.chdir('/content/AI-Exercise-Detection')
print('Working directory:', os.getcwd())

## 4. Generate Synthetic Dataset

This creates training data without needing real exercise videos.

In [ ]:
!python dataset/generate_synthetic.py --samples 300

In [ ]:
import pandas as pd

df = pd.read_csv('dataset/processed_keypoints/keypoints.csv')
print(f'Dataset shape: {df.shape}')
print(f'Classes: {df["label"].nunique()}')
df['label'].value_counts()

## 5. Train the Model

In [ ]:
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Prepare data
X = df.drop(columns=['label']).values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print(classification_report(y_test, y_pred))

In [ ]:
# Save model
os.makedirs('models', exist_ok=True)
joblib.dump(clf, 'models/exercise_model.pkl')
print('Model saved ✅')

## 6. Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title('Exercise Classification — Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Quick Inference Demo

Simulate real-time inference by predicting on a random sample.

In [ ]:
# Load the saved model
model = joblib.load('models/exercise_model.pkl')

# Predict on random samples
for _ in range(5):
    idx = np.random.randint(len(X_test))
    sample = X_test[idx].reshape(1, -1)
    pred = model.predict(sample)[0]
    actual = y_test[idx]
    status = '✅' if pred == actual else '❌'
    print(f'{status}  Predicted: {pred:25s}  |  Actual: {actual}')

## 8. Webcam Demo (Colab)

To run the webcam demo on Colab, you need to use JavaScript to capture
frames from the browser's camera. Below is a helper that does this.

> **Note:** OpenCV `cv2.imshow()` does not work in Colab. This cell
> uses `google.colab.patches.cv2_imshow` instead.

In [ ]:
import sys
sys.path.insert(0, '/content/AI-Exercise-Detection')

from src.pose_detector import PoseDetector
from src.feature_extractor import extract_features
from src.exercise_classifier import ExerciseClassifier
from src.form_analyzer import FormAnalyzer
from src.rep_counter import RepCounter

print('All modules loaded ✅')
print('\nTo run the full webcam demo locally:')
print('  python demo/run_webcam_demo.py')

In [ ]:
# Colab webcam capture helper
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np


def colab_capture_frame():
    """Capture a single frame from the browser webcam."""
    js = Javascript('''
    async function captureFrame() {
      const video = document.createElement('video');
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)
    data = eval_js('captureFrame()')
    binary = b64decode(data.split(',')[1])
    img_array = np.frombuffer(binary, dtype=np.uint8)
    frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    return frame


# Capture & process a single frame
try:
    frame = colab_capture_frame()
    detector = PoseDetector(static_image_mode=True)
    landmarks = detector.get_landmarks(frame)

    if landmarks is not None:
        features = extract_features(landmarks)
        classifier = ExerciseClassifier()
        label, conf = classifier.predict_proba(features)
        exercise, form = ExerciseClassifier.parse_label(label)

        analyzer = FormAnalyzer()
        feedback = analyzer.analyze(exercise, landmarks)

        print(f'Exercise:   {exercise}')
        print(f'Form:       {form}')
        print(f'Confidence: {conf:.2%}')
        print(f'Feedback:   {feedback or "Good form!"}')

        # Show annotated frame
        annotated = detector.draw_landmarks(frame)
        from google.colab.patches import cv2_imshow
        cv2_imshow(annotated)
    else:
        print('No pose detected — try again.')

    detector.close()
except Exception as e:
    print(f'Webcam capture not available in this environment: {e}')
    print('Run the demo locally with: python demo/run_webcam_demo.py')

---

**All done!** 🎉

- Model saved at `models/exercise_model.pkl`
- Run locally: `python demo/run_webcam_demo.py`
- See `README.md` for full documentation